# Notebook 3: The Research Assistant Agent

### Purpose

This is the notebook we have been building toward. You take a research question, the agent plans how to answer it, gathers evidence from your docs and the web, and returns a summary you can trust with citations and claims pointing to a source

By the end you will have built an end-to-end agent workflow and also measure whether the retrieval is good or an assumption.

## What you will build

- Ingestion and chunking
- Semantic retrieval with embeddings (improved version of notebook 2 keyword search)
- Grounded summary with source for each claim
- Retrieval Evaluation with precision @ K

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/03_research_assistant.ipynb)

# STEP 1 - Setup

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothing else in the notebook changes. An agent is a pattern, not a vendor.
#  In this notebook, we install embedding and numeric packages here

# In Google Colab this cell installs the packages. Locally, run once.

%pip install -q anthropic ddgs tavily-python sentence-transformers numpy

import os
import json
from pathlib import Path

REPO_DIR = Path("/content/AGENTIC-AI-Workshop")

if not REPO_DIR.exists():
    !git clone -q https://github.com/ms-cc-org/AGENTIC-AI-Workshop.git /content/AGENTIC-AI-Workshop

%cd /content/AGENTIC-AI-Workshop

PROVIDER = "mock"   # "anthropic" | "mock"
                      # "mock" runs this notebook with NO API key 

MODEL = "claude-haiku-4-5-20251001"  # cheapest current Claude model


if PROVIDER == "anthropic":
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}

_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODEL, max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    raise ValueError(f"PROVIDER must be 'anthropic' or 'mock'; got {PROVIDER!r}")

print(f"Setup ready. PROVIDER={PROVIDER!r}. Commercial provider: Anthropic only.")

# From Notebook 2

Notebook 2 gives us a map where a tool fills gaps for a model by giving it the capabilities it needs to solve tasks. 

In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

# Step 2 - Ingest and Chunk

Whole documents are too big to retrieve well; you want to find the *passage* that answers a question. 
We split each document into overlapping chunks.

**How chunking works here:**

`chunk(text, size=90, overlap=20)` splits a document into windows of 90 words (not characters), where each window shares its last 20 words with the start of the next. The overlap means a sentence that falls near a boundary appears in two chunks, so retrieval can find it from either side.

Each chunk becomes a passage — a dict with three fields:
- id: "filename.md#0", "filename.md#1", ... — the chunk's address, used for citations later
- doc: the source filename
- text: the 90-word window

The id field is what the final briefing will cite. If a citation looks wrong, id tells you exactly which chunk to inspect.

### Data Boundary Warning

Local retrieval does not necessarily mean local processing. When a live provider is selected, retrieved passages are included in the model request and transmitted to that provider. 

Use only **public, synthetic, de-identified,** or **institutionally approved** documents.

The sample documents in `datasets/` are completely synthetic and fictional datasets.
They are safe to use for **this** workshop.

In [ ]:
import glob, os, re, numpy as np
from pathlib import Path

def get_datasets():
    p = Path.cwd()
    while p!= p.parent:
        if (p / "datasets").is_dir():
            return p/ "datasets"
        p = p.parent
    return Path("../datasets")

data_dir = get_datasets()

def load_corpus(folder=data_dir):
    """ Load .md/.txt (readme skipped, unreadable files skipped, explicit encoding)"""
    folder = str(folder)
    docs = {}
    paths = sorted(glob.glob(os.path.join(folder, "*.md")) + glob.glob(os.path.join(folder, "*.txt")))
    for path in paths:
        name = os.path.basename(path)
        if name.lower() == "readme.md":
            continue
        try:
            text = Path(path).read_text(encoding="utf8")
        except (OSError, UnicodeDecodeError) as execp:
            print(f"Skipping {path}: {execp}")
            continue
        docs[name] = text
    if not docs:
        print(f"WARNING: no docs found in {folder!r}. If you are in colab, set repo_url right")
        docs = {"inline_note.md": "Synthetic note: log every query for a reproducible literature triage; measure retrieval precision and recall."}
    return docs

def chunk(text, size = 90, overlap=20):
    """                                                                                                                                                                                                                                         
      Split text into overlapping windows of words.                                                                                                                                                                                             
      size:    window size in words (default 90).  
      overlap: words shared between adjacent windows (default 20).                                                                                                                                                                                
      Returns: list of word-window strings.                       
      """
    words = re.findall(r"\S+", text); out = []; i=0
    while i<len(words):
        out.append(" ".join(words[i:i+size])); i += size-overlap
    return out

CORPUS = load_corpus()
PASSAGES = []
for doc, text in CORPUS.items():
    for x, ch in enumerate(chunk(text)):
        PASSAGES.append({"id": f"{doc}#{x}", "doc": doc, "text":ch})
print(f"datasets from {data_dir}: {len(CORPUS)} documents --> {len(PASSAGES)} passages")

## Step 3 - Embeddings

Embedding turns words (text) into a vector of numbers so that texts with similar meaning land near each other. We compare two vectors with cosine similarity. If the cosine of the angle between them is 1.0 means `very similar` and 0 means unrelated. 

`Anthropic` doesn't have an embedding API. Embeddings stay local and free for this workshop. 

In [ ]:
EMBEDDER = None

def _make_embedder():
    """
    Return an embedding function and its label.
    Tries sentence-transformers first; falls back to TF-IDF if not installed.
    Returns: (callable(texts) -> np.array, label string)                                                                                                                                                                                            
    """ 
    # 1) local sentence-transformers
    try:
        from sentence_transformers import SentenceTransformer
        m=SentenceTransformer("all-MiniLM-L6-v2")
        return (lambda texts: np.array(m.encode(list(texts)), dtype=float)), "sentence-transformers:all-MiniLM-L6-v2"
    except Exception:
        pass
    # 2) TF-IDF fallback (transparent, will always work, weaker)
    import math
    vocab = {}
    for p in PASSAGES:
        for w in set(re.findall(r"[a-z]+", p["text"].lower())): vocab[w]=vocab.get(w,0)+1
    N=len(PASSAGES); idf={w: math.log((N+1)/(df+1))+1 for w,df in vocab.items()}
    terms=sorted(idf)
    index={w:i for i,w in enumerate(terms)}
    def emb(texts):
        M=np.zeros((len(texts), len(terms)))
        for r,t in enumerate(texts):
            ws=re.findall(r"[a-z]+", t.lower())
            for w in ws:
                if w in index: M[r, index[w]] += idf[w]
        return M
    return emb, "tfidf-fallback (install sentence-transformers for real embeddings)"

EMBEDDER, EMBED_NAME = _make_embedder()
print("Using embedder:", EMBED_NAME)

# Embed every passage once.
PASSAGE_VECS = EMBEDDER([p["text"] for p in PASSAGES])
print("passage matrix shape:", PASSAGE_VECS.shape)

In [ ]:
def cosine(a, b):
    return float(a @ b / ((np.linalg.norm(a)*np.linalg.norm(b)) + 1e-9))

def retrieve(query, k=3):
    #Return the k passages most similar to the query, with scores
    qv = EMBEDDER([query])[0]
    scored = sorted(
        ({"passage":p, "score":cosine(qv, v)} for p,v in zip(PASSAGES, PASSAGE_VECS)),
        key=lambda x: x["score"], reverse=True)
    return scored[:k]

for hit in retrieve("When is the NSF AI proposal deadline?", k=2):
    print(f"{hit['score']:.3f}  {hit['passage']['id']}: {hit['passage']['text'][:90]}...")

## Step 4 Measuring retrieval before you trust it

A summary is only right, if the retrieval returns the exact passages. Most people skip this and it is the one that separates a demo from a research tool. So we score it. Precision@k = of the passages we returned, what fraction are actually relevant?


In [ ]:
# Tiny hand-labeled eval set: query -> the document that should be retrieved.
EVAL = [
    ("NSF AI proposal deadline and award size", "NSF_AI_Funding_2026.md"),
    ("can I use AI on human-subjects data", "irb_policy_ai.md"),
    ("how to make a literature review reproducible", "literature_review_methods.md"),
    ("does RAG stop hallucination", "RAG_grounding_note.md"),
]
def precision_at_k(k=5):
    scores=[]
    for query, ref_doc in EVAL:
        hits=retrieve(query, k=k)
        rel=sum(1 for h in hits if h["passage"]["doc"]==ref_doc)/k
        scores.append(rel)
        print(f"  p@{k}={rel:.2f}  '{query[:40]}...' -> top: {hits[0]['passage']['doc']}")
    print(f"\nMean precision@{k}: {sum(scores)/len(scores):.2f}  (embedder: {EMBED_NAME})")
    return sum(scores)/len(scores)

# Only runs if the labeled docs are present (the shipped synthetic corpus).
if any(p['passage']['doc'] in {g for _,g in EVAL} for p in [{'passage':x} for x in PASSAGES]):
    _ = precision_at_k(k=3)
else:
    print("Eval skipped: shipped datasets not found (you swapped in your own).")

## Step 5 - Full Agent: Plan, Gather, Synthesize

Now the full agent. There are 3 visible stages:

1. **Plan** - the model breaks the question into sub-questions
2. **Gather** - for each sub-question, retrieve passages
3. **Synthesize** - Write a summary which includes citations from passage it came from. To guard against hallucinations, we'll instruct the model to answer *only* from the gathered evidence. 

**What the next cell defines**:

| Function | Stage | What it does |
| --- | --- | ---|
| `plan(question)` | Plan | Calls the model with a planner system prompt. returns a list of sub-questions, one per line |
| `gather(subquestions, k, use_web)` | Gather | Runs `retrieve()` for each sub-question, de-duplicates passage IDs and returns a ranked list |
| `synthesize(question, evidence)` | Synthesize | Calls the model with system prompt. returns the cited briefing text |
| `research_assistant(...)` | All three | Runs plan --> gather --> synthesize. This is the function you call directly in the demo cell following the next cell | 
| `build_evidence_table (..)` | Post-synthesis | Turns the passage list into a structured table with source_id, doc, claim, review_status |
| `validate_evidence(table)` | Post-synthesis | Checks each table record and returns a list of problems |

In [ ]:
PLANNER = ("You are a research planner. Break the user's question into 2-4 focused sub-questions. Return ONE sub-question per line, no numbering.")
SYNTH = ("You are a careful research assistant. Using ONLY the evidence passages provided, write a short briefing. "
            "After each claim, cite the passage id in [brackets]. If the evidence does not cover something, say so. Do not invent facts.")

def plan(question):
    reply = call_llm([{"role":"user","content":question}], system=PLANNER)
    subs = [l.strip("-• ").strip() for l in reply["text"].splitlines() if l.strip()]
    return subs or [question]

def gather(subquestions, k=2, use_web=False):
    evidence=[]
    for sq in subquestions:
        for hit in retrieve(sq, k=k):
            evidence.append(hit["passage"])
        if use_web:
            # plug in search_web from Notebook 2 here if you want current facts
            pass
    # de-dup by id, preserve order
    seen=set(); uniq=[]
    for p in evidence:
        if p["id"] not in seen: seen.add(p["id"]); uniq.append(p)
    return uniq

def synthesize(question, evidence):
    block="\n\n".join(f"[{p['id']}] {p['text']}" for p in evidence)
    prompt=f"Question: {question}\n\nEvidence passages:\n{block}\n\nWrite the cited briefing."
    return call_llm([{"role":"user","content":prompt}], system=SYNTH)["text"]

def research_assistant(question, k=2, use_web=False, verbose=True):
    subs = plan(question)
    if verbose: print("PLAN:"); [print(" -", s) for s in subs]
    ev = gather(subs, k=k, use_web=use_web)
    if verbose: print(f"\nGATHERED {len(ev)} passages:", [p['id'] for p in ev])
    report = synthesize(question, ev)
    return report, subs, ev

def build_evidence_table(question, evidence):
    #Turn gathered passages into a structured table with source_id, doc, claim, and review_status
    return {"question": question, "records": [
        {"source_id": p["id"], "doc": p["doc"],
         "claim": p["text"][:160], "review_status": "unreviewed"} for p in evidence]}

def validate_evidence(table):
    """Two layers: valid (right fields) is not the same as trustworthy.
    Trust here = a source_id traceable to a chunk, and a non-empty claim."""
    problems=[]
    for i, r in enumerate(table.get("records", [])):
        if set(r) != {"source_id","doc","claim","review_status"}:
            problems.append(f"record {i}: wrong fields")
        if "#" not in r.get("source_id",""):
            problems.append(f"record {i}: source_id not traceable to a chunk")
        if not r.get("claim","").strip():
            problems.append(f"record {i}: empty claim")
    return problems

## A note on mock mode for this section

`research_assistant()` calls `call_llm()` twice internally, once inside plan() and synthesize(). Both calls go through the same mock queue.

So the `mock_reset([...])` below loads two entries:
1. First "final" --> consumed by `plan()` and its text is split into sub-questions
2. Second "final" --> consumed by `synthesize()` and its text becomes the cited briefing

This is why the output has two visible stages (a plan list, then a briefing) even though only one mock_reset call was made.

With a live provider: `mock_reset` is ignored. The model generates its own sub-questions in `plan()` and its own cited synthesis in `synthesize()`. Outputs will vary across runs.

In [ ]:
QUESTION = "What do I need to know to apply for the NSF AI infrastructure grant using AI tools responsibly?"

# Mock script: a realistic plan + a grounded, cited synthesis (real provider does this itself).
mock_reset([
    ("final", "What is the NSF AI infrastructure deadline and award size?\n"
              "What are the rules for using AI on human-subjects data?\n"
              "How do I keep AI-assisted work reproducible?"),
    ("final",
     "Briefing:\n"
     "--> The NSF AI infrastructure program funds up to $600,000 over three years; full proposals are due November 3, 2026 [NSF_AI_Funding_2026.md#0].\n"
     "--> AI tools may not process identifiable human-subjects data unless the vendor is covered by an approved data agreement; free consumer AI is not approved for regulated data [irb_policy_ai.md#0].\n"
     "--> For reproducibility, log every query and record which model and version processed the data [literature_review_methods.md#0]. The evidence does not cover indirect cost rates."),
])

report, subs, ev = research_assistant(QUESTION, k=2)
print("\n" + "-"*50 + "\nCITED BRIEFING:\n" + "-"*50)
print(report)

# The briefing is for humans; the evidence table is the structured, validated companion. Nothing is "checked" yet.
table = build_evidence_table(QUESTION, ev)
problems = validate_evidence(table)
print("\n" + "-"*50 + "\nVALIDATED EVIDENCE TABLE:\n" + "-"*50)
print("valid:", not problems, "| problems:", problems or "none")
print(f"{len(table['records'])} records, each review_status='unreviewed' until a human checks it")

## Step 6 - Exercise

This is where the workshop becomes your own.

1. Drop one or two of *your own* documents (`.md`/`.txt`) into the `datasets/` folder. In Colab, upload them and adjust the folder path.
2. Re-run Steps 2–3 to re-ingest and re-embed.
3. Set `QUESTION` to something you actually want answered from the document.
4. Set a real `PROVIDER` and key, and run `research_assistant(QUESTION)`.
5. **Check the citations by hand.** Open one cited passage. Does it really say what the briefing claims?

**The honest test:** if a citation doesn't support its claim, the agent hallucinated. Note where and why. That's the most useful thing you'll learn today.

In [ ]:
# YOUR QUESTION HERE (needs a real PROVIDER + API key, or keep a mock script):
MY_QUESTION = "Summarize what our local policies say about using AI on sensitive data."
mock_reset([
    ("final", "What data is off-limits for AI?\nWhen is human review required?"),
    ("final", "Local policy: AI may not process identifiable human-subjects data without an approved "
              "vendor agreement, and human review is required before AI output affects a participant "
              "[irb_policy_ai.md#0]."),
])
rep, s, e = research_assistant(MY_QUESTION, k=2)
print(rep)

## Reflection
- Where did retrieval help, and where did it miss?
- Was every citation right when you checked it?
- What document collection of your own would make this genuinely useful next week?